# Company Sales Brochure Generator

Create a product which builds a **sales brochure** for a company to be used for prospective clients, investors and potential recruits.

**Input:** company name and their main website

**Tools & Techniques:**
- OpenAI API (Chat Completions API)
- One-shot prompting
- Streaming results as markdown, json
- Convert this into a commercial solution

It will involve pulling multiple web pages: by starting with the primary webpage, scanning it for relavant links, and then pulling the content of those links.

Will include compounding LLM calls (multiple LLM calls) to get better output.

**GOAL:** Work as an AI Engineer, and build a solution using GPT models.

In [6]:
# imports
import os
import json
from IPython.display import display, Markdown
from openai import OpenAI
from dotenv import load_dotenv

In [7]:
# load openai api key
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_TUTORIALS_API_KEY")

if not "sk-proj" in OPENAI_API_KEY:
  raise ValueError("Please set your OpenAI API key in the .env file")
else:
  print("OpenAI API key loaded successfully")

OpenAI API key loaded successfully


In [8]:
llm = OpenAI(api_key=OPENAI_API_KEY)

In [9]:
# function to scrape a website and get all its links (anchor tags)
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

def fetch_website_links(url):
    """
    Scrape a website and return a list of all links found in the webpage.
    
    Args:
        url (str): The URL of the website to scrape.
    
    Returns:
        list: A list of absolute URLs found in the webpage.
    """
    try:
        response = requests.get(url)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')
        links = []
        for a_tag in soup.find_all('a', href=True):
            link = urljoin(url, a_tag['href'])
            links.append(link)
        return links
    except requests.RequestException as e:
        print(f"Error fetching the website: {e}")
        return []
    except Exception as e:
        print(f"An error occurred: {e}")
        return []

In [10]:
# testing out this function
test_links = fetch_website_links("https://www.sarvam.ai/")
# https://simplysouthindia.com/
unique_links = set(test_links)
print(len(unique_links))
for link in unique_links:
    print(link)

22
https://discord.com/invite/5rAsykttcs
https://www.sarvam.ai/blogs/sarvam-30b-105b
https://www.sarvam.ai/api-pricing
https://youtube.com/@sarvamai
https://www.sarvam.ai/
https://dashboard.sarvam.ai/
https://www.sarvam.ai/blogs/evaluating-indian-language-asr
https://www.sarvam.ai/privacy-policy
https://www.sarvam.ai/about-us
https://www.sarvam.ai/apis/document-digitisation
https://www.sarvam.ai/stories/tata-capital-ai-voice-transformation
https://www.sarvam.ai/apis/speech-to-text
https://www.sarvam.ai/terms-of-use
https://www.sarvam.ai/blogs
https://www.sarvam.ai/products/conversational-agents
https://x.com/sarvamai
https://www.sarvam.ai/careers
https://www.linkedin.com/company/sarvam-ai
https://www.sarvam.ai/products/akshar
https://www.sarvam.ai/blogs/introducing-indus
https://www.sarvam.ai/apis/text-to-speech
https://www.sarvam.ai/products/studio


## Figure out the relavant links using an LLM

Invoke an LLM to decide what links are relavant. Doing this manually for a lot of links is hard.

In the below code, we will let LLM **output data in a structured format**. Additionally, we will use **one-shot prompting** technique, where we provide one example of what the output should look like.

Think of the following:
1. System Prompt (Instructions to the LLM)
2. User Prompt (Input to the LLM)

**Why output in JSON?**
- LLMs are trained on lots of data in one of these formats:
  - Plain english
  - Markdown
  - JSON
- Using structured input / output format, LLMs behave more coherently. Its a good way to express information and expect good results.

**Giving examples in the prompt:**
- One shot prompting: give one example of how the output should look like.
- Multi shot prompting: give multiple examples of how the output should look like.
- Zero shot prompting: don't give any examples of how the output should look like.

In [11]:
system_prompt = """
You are an expert AI agent who can look at a given set of links, and decide which ones are relevant 
to be included in a brochure to a company where the target audience could be prospective clients, investors, recruits, etc.

The useful links could be like the about page, careers page, products page, etc.

The links should be returned in a JSON format as follows:
{
  "links": [
    {"name": "About Us", "url": "https://example.com/about"},
    {"name": "Careers", "url": "https://example.com/careers"},
    {"name": "Products", "url": "https://example.com/products"}
  ]
}
"""

In [12]:
# construct the user prompt, which contains the list of links as well
def generate_user_prompt(website_name, url):
  links = fetch_website_links(url)
  user_prompt = f"""
Below are the list of links found in the website of {website_name} at the URL {url}.
Your task is to decide what links are relavant to create a brochure for the company, where the potential stake holders are prospective clients, investors, recruits, etc.
Do not include links about non-relavant items like Terms of Service, Privacy Policy, emails, links to youtube, etc. But links to other social media like Twitter, LinkedIn, Instagram might be relavant ones.

Links:
{"\n".join(links)}
  """
  return user_prompt

In [13]:
print(generate_user_prompt("Sarvam.AI", "https://www.sarvam.ai/"))


Below are the list of links found in the website of Sarvam.AI at the URL https://www.sarvam.ai/.
Your task is to decide what links are relavant to create a brochure for the company, where the potential stake holders are prospective clients, investors, recruits, etc.
Do not include links about non-relavant items like Terms of Service, Privacy Policy, emails, links to youtube, etc. But links to other social media like Twitter, LinkedIn, Instagram might be relavant ones.

Links:
https://www.sarvam.ai/
https://www.sarvam.ai/
https://dashboard.sarvam.ai/
https://www.sarvam.ai/stories/tata-capital-ai-voice-transformation
https://www.sarvam.ai/blogs/evaluating-indian-language-asr
https://www.sarvam.ai/blogs/sarvam-30b-105b
https://www.sarvam.ai/blogs/introducing-indus
https://www.sarvam.ai/blogs
https://www.sarvam.ai/
https://www.sarvam.ai/products/conversational-agents
https://www.sarvam.ai/products/studio
https://www.sarvam.ai/products/akshar
https://www.sarvam.ai/apis/text-to-speech
https

In [14]:
# function to invoke the LLM to filter out the relevant links

def filter_relavant_links_for_brochure(website_name, url):
  response = llm.chat.completions.create(
    model="gpt-5-nano",
    messages=[
      {"role": "system", "content": system_prompt},
      {"role": "user", "content": generate_user_prompt(website_name, url)}
    ],
    response_format={"type": "json_object"} # return the response as a JSON object
  )
  print("Response:")
  print(response, end="\n-----\n")
  json_string = response.choices[0].message.content
  print("JSON string:")
  print(json_string, end="\n-----\n")
  json_response = json.loads(json_string)
  return json_response

`response_format={"type": "json_object"}` will force the model to predict the tokens such that the overall output will be a valid JSON.

In [15]:
relavant_links = filter_relavant_links_for_brochure("Sarvam.AI", "https://www.sarvam.ai/")

Response:
ChatCompletion(id='chatcmpl-DTjYfZoMypwEujxVz3AkqD1EKhBQr', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n  "links": [\n    {"name": "Home", "url": "https://www.sarvam.ai/"},\n    {"name": "About Us", "url": "https://www.sarvam.ai/about-us"},\n    {"name": "Careers", "url": "https://www.sarvam.ai/careers"},\n    {"name": "Blogs", "url": "https://www.sarvam.ai/blogs"},\n    {"name": "Case Study: Tata Capital AI Voice Transformation", "url": "https://www.sarvam.ai/stories/tata-capital-ai-voice-transformation"},\n    {"name": "Conversational Agents", "url": "https://www.sarvam.ai/products/conversational-agents"},\n    {"name": "Studio", "url": "https://www.sarvam.ai/products/studio"},\n    {"name": "Akshar", "url": "https://www.sarvam.ai/products/akshar"},\n    {"name": "Text to Speech API", "url": "https://www.sarvam.ai/apis/text-to-speech"},\n    {"name": "Speech to Text API", "url": "https://www.sarvam.ai/apis/speech-t

In [16]:
langchain_relavant_links = filter_relavant_links_for_brochure("Langchain", "https://www.langchain.com/")

Response:
ChatCompletion(id='chatcmpl-DTjZ2vT0Y4CwomU3MlnMorHanvb4n', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n  "links": [\n    {"name": "About LangChain", "url": "https://www.langchain.com/about"},\n    {"name": "Careers", "url": "https://www.langchain.com/careers"},\n    {"name": "Startups", "url": "https://www.langchain.com/startups"},\n    {"name": "Pricing", "url": "https://www.langchain.com/pricing"},\n    {"name": "Customers", "url": "https://www.langchain.com/customers"},\n    {"name": "Resources", "url": "https://www.langchain.com/resources"},\n    {"name": "Documentation", "url": "https://docs.langchain.com/"},\n    {"name": "Academy", "url": "https://academy.langchain.com/"},\n    {"name": "LangChain Core", "url": "https://www.langchain.com/langchain"},\n    {"name": "LangGraph", "url": "https://www.langchain.com/langgraph"},\n    {"name": "LangSmith Platform", "url": "https://www.langchain.com/langsmith-platfor

## Scrape content from all the relavant links

Scrape all relavant links, get their content, and glue together all the content.

This is the input to the LLM when generate the brochure.

In [17]:
# function to scrape content of a given link
def fetch_website_content(url):
    """
    Scrape a website and return its content as text.
    
    Args:
        url (str): The URL of the website to scrape.
    
    Returns:
        str: The text content of the website.
    """
    try:
        response = requests.get(url)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')
        # Remove script and style elements
        for script in soup(["script", "style"]):
            script.extract()
        # Get text
        text = soup.get_text()
        # Clean up whitespace
        lines = (line.strip() for line in text.splitlines())
        chunks = (phrase.strip() for line in lines for phrase in line.split("  "))
        text = '\n'.join(chunk for chunk in chunks if chunk)
        return text
    except requests.RequestException as e:
        print(f"Error fetching the website: {e}")
        return ""
    except Exception as e:
        print(f"An error occurred: {e}")
        return ""

In [18]:
# test this out

content = fetch_website_content(relavant_links['links'][0]['url'])
print(content)

Sarvam | India's Full-Stack Sovereign AI Platform
PlatformDevelopersResourcesCompanyExperience SarvamTalk to Sales
India's Sovereign AI PlatformAI for all from IndiaBuilt on sovereign compute. Powered by frontier-class models.Delivering population-scale impact.Experience Sarvam
India builds with Sarvam
Powering India's AI-first futureSovereign by designBuild, deploy, and run AI with full control, developed and operated entirely in IndiaState of the art modelsIndustry-leading models built for India's languages, culture, and contextHuman at the coreForward deployed engineers work alongside your teams to deliver production-ready agents
For Enterprise | Government | DevelopersIndia's full-stack sovereign AI platformPopulation-scale ApplicationsBuilding products India can use. Conversational agents fluent in India's languages. Platforms that run enterprise workflows from start to finish.SamvaadStudioState-of-the-art ModelsState-of-the-art models trained on sovereign data, delivering strong 

In [19]:
# pretty print relavant links JSON for visualization
print(json.dumps(relavant_links, indent=2))

{
  "links": [
    {
      "name": "Home",
      "url": "https://www.sarvam.ai/"
    },
    {
      "name": "About Us",
      "url": "https://www.sarvam.ai/about-us"
    },
    {
      "name": "Careers",
      "url": "https://www.sarvam.ai/careers"
    },
    {
      "name": "Blogs",
      "url": "https://www.sarvam.ai/blogs"
    },
    {
      "name": "Case Study: Tata Capital AI Voice Transformation",
      "url": "https://www.sarvam.ai/stories/tata-capital-ai-voice-transformation"
    },
    {
      "name": "Conversational Agents",
      "url": "https://www.sarvam.ai/products/conversational-agents"
    },
    {
      "name": "Studio",
      "url": "https://www.sarvam.ai/products/studio"
    },
    {
      "name": "Akshar",
      "url": "https://www.sarvam.ai/products/akshar"
    },
    {
      "name": "Text to Speech API",
      "url": "https://www.sarvam.ai/apis/text-to-speech"
    },
    {
      "name": "Speech to Text API",
      "url": "https://www.sarvam.ai/apis/speech-to-text"

In [20]:
def generate_website_content_context(website_name: str, links: list) -> str:
  """
  Input: website_name - string
  Input: links - list of dictionaries with fields: name, url
  """
  content = """
  Website Name: {website_name}

  Relavant Links & their Content:

  """
  for link in links:
    link_content = fetch_website_content(link['url'])
    content += f"""
    --------------

    Link Name: {link['name']}
    Link URL: {link['url']}
    Link Content: 
    {link_content}

    """
  return content

In [21]:
company_context = generate_website_content_context("Sarvam.AI", filter_relavant_links_for_brochure("Sarvam.AI", "https://www.sarvam.ai/")['links'])

print(company_context)

Response:
ChatCompletion(id='chatcmpl-DTjZVftT9KQk419usqoAk7gtgEPRN', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n  "links": [\n    { "name": "Home", "url": "https://www.sarvam.ai/" },\n    { "name": "About Us", "url": "https://www.sarvam.ai/about-us" },\n    { "name": "Careers", "url": "https://www.sarvam.ai/careers" },\n    { "name": "Products: Conversational Agents", "url": "https://www.sarvam.ai/products/conversational-agents" },\n    { "name": "Products: Studio", "url": "https://www.sarvam.ai/products/studio" },\n    { "name": "Products: Akshar", "url": "https://www.sarvam.ai/products/akshar" },\n    { "name": "API: Text-to-Speech", "url": "https://www.sarvam.ai/apis/text-to-speech" },\n    { "name": "API: Speech-to-Text", "url": "https://www.sarvam.ai/apis/speech-to-text" },\n    { "name": "API: Document Digitisation", "url": "https://www.sarvam.ai/apis/document-digitisation" },\n    { "name": "API Pricing", "url": "http

In [22]:
# explore the context length and number of tokens
import tiktoken

print(f"Number of characters in context: {len(company_context)}")
print(f"Number of words in context: {len(company_context.split())}")

enc = tiktoken.encoding_for_model("gpt-5-nano")

print(f"Number of tokens: {len(enc.encode(company_context))}")


Number of characters in context: 142595
Number of words in context: 18653
Number of tokens: 31926


## Generate the brochure by invoking the LLM

In [23]:
brochure_gen_system_prompt = """
You are an expert in creating engaging & energetic short brochures for companies (for prospective clients, investors, recruits, etc.) based on information in relavant pages of the company's website.

Create the brochure and respond in Markdown format.

Include emojis to keep content engaging.
"""

In [24]:
def construct_brochure_gen_user_prompt(website_name, primary_url):
  extracted_relavant_pages = filter_relavant_links_for_brochure(website_name, primary_url)
  return f"""
You are looking at a company called: {website_name}

Below is the content of various relavant links of the company.

Based on this knowledge, create a short brochure and output in Markdown format. Do not include additional chat messages.

Company Details:
{generate_website_content_context(website_name, extracted_relavant_pages['links'])}
  """

In [25]:
def create_brochure(website_name, primary_url):
  response = llm.chat.completions.create(
    model="gpt-5-nano",
    messages=[
      {"role": "system", "content": brochure_gen_system_prompt},
      {"role": "user", "content": construct_brochure_gen_user_prompt(website_name, primary_url)}
    ]
  )
  response_text = response.choices[0].message.content
  # display the brochure in markdown
  display(Markdown(response_text))

In [26]:
create_brochure("Sarvam.AI", "https://www.sarvam.ai/")

Response:
ChatCompletion(id='chatcmpl-DTja042jcWRK7RlFdRnLXOM2ShGtW', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n  "links": [\n    {"name": "Home", "url": "https://www.sarvam.ai/"},\n    {"name": "About Us", "url": "https://www.sarvam.ai/about-us"},\n    {"name": "Careers", "url": "https://www.sarvam.ai/careers"},\n    {"name": "Products: Conversational Agents", "url": "https://www.sarvam.ai/products/conversational-agents"},\n    {"name": "Studio", "url": "https://www.sarvam.ai/products/studio"},\n    {"name": "Akshar", "url": "https://www.sarvam.ai/products/akshar"},\n    {"name": "APIs: Text to Speech", "url": "https://www.sarvam.ai/apis/text-to-speech"},\n    {"name": "APIs: Speech-to-Text", "url": "https://www.sarvam.ai/apis/speech-to-text"},\n    {"name": "APIs: Document Digitisation", "url": "https://www.sarvam.ai/apis/document-digitisation"},\n    {"name": "API Pricing", "url": "https://www.sarvam.ai/api-pricing"},\n  

# Sarvam AI — AI for India 🇮🇳

Experience India's sovereign, full‑stack AI platform designed for languages, culture, and context across the nation. Built in India. Operated in India. Sovereign by design. 🧭✨

---

## Why Sarvam

- India's sovereign AI platform built on sovereign compute with frontier‑class models. 🧠💡
- End‑to‑end stack: foundation models, deployment, governance, and AI tooling all in‑country. 🇮🇳
- Language and cultural focus: models trained for India's languages, dialects, and contexts. 🗣️🌐
- Enterprise, government, and developer friendly — secure, scalable, and production‑ready. 🔐⚙️

---

## The Platform

- Sovereign by design: run AI where you operate — cloud, private cloud (VPC), or on‑premises. ☁️🔒🏢
- Enterprise‑grade security baked in from day one. SOC‑2 compliant, end‑to‑end encryption. 🛡️
- Fast time‑to‑value: fully managed, scalable infrastructure with a focus on reliability and governance. 🚀

---

## Core Products

- Sarvam Samvaad — Conversational Agents
  - Full‑stack platform to deploy AI agents across voice, WhatsApp, and web surfaces. 🔊💬
  - Real‑time, multilingual, 11 Indian languages plus English. Deep enterprise integrations. 📈
  - Multi‑agent orchestration for complex workflows. 🧩

- Sarvam Studio — AI Dubbing & Translation for Indian Languages
  - Multilingual content transformation: translate, dub, and ship content across 11+ Indian languages. 🎬🎤
  - Production‑grade output with precise audio‑visual sync and quality checks. SOC 2 compliant. 🔒🎧

- Sarvam Akshar — Document Digitisation
  - Intelligent OCR for Indian documents: reads complex layouts, dense tables, and Indic scripts. 🗂️🧾
  - Output in HTML, JSON, or Markdown with layout and reading order preserved. 23 languages supported. 🗺️
  - API access for batch workflows; visual platform for review and correction. 🧩

- APIs & Tools
  - Text to Speech (Bulbul) — natural Indian‑language voices, 11 languages, 35+ voices. 🗣️🎙️
  - Speech to Text (Saaras) — robust transcription for 22 Indian languages with code‑mix support. 🗨️📝
  - Document Digitisation API — async processing, 23 languages, HTML/Markdown output. 📄⚡
  - Translation, Transliteration, Language Identification, and more — all on a single platform. 🌐🔤

---

## Real‑World Impact

- Enterprise‑grade deployments that scale conversations and automate workflows across customer journeys.  
- Tata Capital case study: multilingual voice AI enabling highly personalized, cost‑effective interactions and stronger engagement across loan products. “Voice‑driven AI is reshaping customer engagement … with multilingual interactions across our consumer loan products.” 💬🤝

---

## Open Source & Indus

- Open‑source momentum: Sarvam 30B and 105B models released—solid open foundation for sovereign AI. 🧰🔓
- Indus — an interface to experience our 105B models in limited beta, built for accuracy, efficiency, and Indian alignment. 🧭💡

---

## API Pricing & Getting Started

- Transparent pay‑per‑use pricing. Every plan includes ₹1,000 in free credits to start. 🎁💳
- Core API prices (illustrative overview):
  - Sarvam Vision (document & image analysis): ₹1.50 per page. 🧾
  - Text to Speech (Bulbul v3): ₹30 per 10,000 characters. 🔊
  - Speech to Text (Saaras): ₹30 per hour of audio (base); higher with diarization/translation options. 🧠🗣️
  - Language services (Translate, Transliterate, Language ID): tiered per usage. 🌐
- Plans:
  - Starter (pay‑as‑you‑go): 60 requests/minute, community support. 🚦
  - Pro: higher rate limits, email support. 🔧
  - Business: production workloads, solutions engineer, Slack support. 🛠️
- Access: Start for free with credits; no credit card required upfront. Instant API keys; easy onboarding. 🚀🔑
- Languages: 23 languages supported for document digitisation; 22 Indian languages plus English for speech & ASR; TTS covers 11 Indian languages plus English accents. 🗺️🗣️

- Ready to build? Get started and explore the API dashboard, or join the Studio waitlist to experience Samvaad Studio in beta. 🧪🎯

---

## What Makes Sarvam Different

- Sovereign stack: everything developed, deployed, and governed in India. 🇮🇳
- Language‑first AI: models trained on Indian languages and dialects for natural, contextually aware interactions. 🗣️💬
- Production‑grade tooling: high‑throughput, low latency, secure integrations with enterprise systems. ⚡🔒
- Community & ecosystem: open models, Indus interface, and robust evaluation tooling to accelerate Indian AI innovation. 🌱

---

## Get In Touch

- Learn more: our products, research, and API offerings are designed to empower enterprises, governments, and developers to build with full agency. 📚🔗
- Follow: LinkedIn, X (Twitter), YouTube, and Discord for updates, case studies, and developer resources. 📢🌐

---

Build the future of India's AI with Sarvam. AI for India starts here. 🚀🇮🇳

In [27]:
create_brochure("Langchain", "https://www.langchain.com/")

Response:
ChatCompletion(id='chatcmpl-DTjalHhdRTuDRRpEU5htSyVJLZDzf', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n  "links": [\n    {"name": "About LangChain", "url": "https://www.langchain.com/about"},\n    {"name": "Careers", "url": "https://www.langchain.com/careers"},\n    {"name": "Startups", "url": "https://www.langchain.com/startups"},\n    {"name": "LangChain Partner Network", "url": "https://www.langchain.com/langchain-partner-network"},\n    {"name": "Pricing", "url": "https://www.langchain.com/pricing"},\n    {"name": "Contact Sales", "url": "https://www.langchain.com/contact-sales"},\n    {"name": "LangSmith Platform", "url": "https://www.langchain.com/langsmith-platform"},\n    {"name": "Deep Agents", "url": "https://www.langchain.com/deep-agents"},\n    {"name": "LangChain (Product)", "url": "https://www.langchain.com/langchain"},\n    {"name": "LangGraph", "url": "https://www.langchain.com/langgraph"},\n    {"na

# LangChain — The Agent Engineering Platform 🚀

LangChain is on a mission to unlock the future of AI agents. By combining open-source frameworks with an enterprise-grade observability, evaluation, and deployment platform, LangChain helps developers ship reliable agents faster — with data-backed confidence and zero vendor lock-in. 🧭✨

---

## Why LangChain?

- LLMs are powerful, but truly valuable when they can act on data and take actions through agents. LangChain provides the tools to build, test, observe, and deploy those agents at scale. 🧠➡️🤖
- Open-source roots, backed by a commercial platform (LangSmith) to accelerate real-world production. Open, extensible, and designed for rapid iteration. 🔧🌐
- A comprehensive toolkit spanning long-running agents, orchestration, memory, human-in-the-loop, and production-grade runtimes.

---

## The Platform & Frameworks (What you get)

- **LangSmith Platform** — AI agent observability, evaluation, deployment, and fleet management in one place.
  - Observability: See exactly what your agents do with full traces, dashboards, and a built-in assistant to speed debugging. 🕵️‍♀️📈
  - Evaluation: Run online/offline evals, calibrate judges, and compare results to prevent regressions. 🎯🧪
  - Deployment: One-click deployments, versioning, rollbacks, and centralized governance across the enterprise. 🚀🗂
  - Fleet: No-code/low-code agent creation for non-technical teams to automate everyday tasks. 🧰🤖

- **Deep Agents** — An open-source harness for long-running, autonomous agents.
  - Planning, memory, context management, and multi-agent orchestration out of the box. 🧩🧭
  - Native traceability, memory across sessions, and middleware to optimize latency and cost. ⏱💾

- **LangChain** — Open-source AI agent framework.
  - MIT-licensed, model/tool integration with 1000+ integrations, and durable runtimes.
  - Templates and components to ship agents quickly, with vendor-neutral design. 🧰🔗

- **LangGraph** — Agent orchestration framework for reliable, low-level control.
  - Expressive, customizable workflows (single/multi-hub, hierarchical) with strong memory and streaming support. 🧠⚙️
  - Human-in-the-loop checks to steer actions and prevent drift. 🧑‍⚖️🛡

- **LangSmith Fleet** — No-code agents for your whole team.
  - Describe what you need; Fleet builds the agent, learns from feedback, and asks before taking sensitive actions. 🧪🧭

- **LangSmith Platform, Observability, Evaluation, Deployment, Fleet** — All integrated to observe, measure, deploy, and scale agent workloads across your organization. 📊🏢

- All frameworks are designed to work with any model provider, with durable runtimes, memory, and streaming to enhance UX and reliability. 🧬💡

---

## Enterprise-Grade Capabilities

- Deploy real-world agent interactions with exact-once execution, multi-agent coordination, and human-in-the-loop approvals. 🔄🧩
- Horizontal scaling for long-running, bursty workloads, with centralized registry and versioning. 🧭⚡
- Flexible hosting options: SaaS with data residency (US/EU), hybrid, or self-hosted deployments. 🗺🔒
- Security and governance: SSO, RBAC/ABAC, data encryption, audit logs, and architectural guidance. 🛡🔐

---

## Startups, Partners, & Global Reach

- LangSmith for Startups: observe, evaluate, and deploy with discounted pricing and credits to help early-stage builders win fast. 🚀💼
- Startup tiers and benefits include LangSmith credits, exclusive events, and coworking with LangChain leadership. 🎟🤝
- LangChain Partner Network: cloud, consulting, and services partners to help you deploy at scale.
  - Cloud partners: AWS, Microsoft Azure, Google Cloud Platform. ☁️🗺
  - Consulting/Services partners: Caylent, Parlance Labs, Lightci, Tenex, Quantiphi, and more. 🤝💡
- LangChain is trusted by top teams and 35% of the Fortune 500, with 1B+ open-source downloads and 1B+ events processed daily on LangSmith. 💼🏢🌐

---

## Real-World Impact

- Proven by customer stories across industries: Pigment, Rakuten, PagerDuty, Podium, Rippling, ServiceNow, Monday.com, C.H. Robinson, Cisco, Vodafone, Trellix, and more. 📚💬
- LangSmith provides end-to-end visibility, evaluation, and deployment to turn traces into actionable improvements. 🧭➡️ improvement

- The LangChain ecosystem includes Academy courses to upskill teams, a vibrant community, and comprehensive documentation to accelerate your journey. 🎓🧑‍🏫

---

## How to Get Started

- Start building today: 1-click deployments, templates, and tools to ship agents faster. 🏁✨
- Get a demo: See LangSmith in action and walk through a tailored tour of your use case. 📞🎬
- For startups: apply for LangSmith for Startups and unlock discounted tiers, credits, and exclusive programs. 🧾🎯
- Learn and grow: LangChain Academy and in-depth docs to level up your agent engineering skills. 🧠📚

- Community and support: join the LangChain Community Slack, forums, and support portals to connect with fellow builders. 💬🌐

---

## Quick Take

- The platform combines open-source frameworks (LangChain, LangGraph, Deep Agents) with LangSmith, a production-grade agent observability/evaluation/deployment suite. 🧭🛠
- Designed for teams building reliable, scalable, and memory-rich agents with human-in-the-loop where needed. 🧠🤖🧑‍⚖️
- Ready to ship reliable agents faster? Get started with a demo, start free, or explore startup and enterprise options. 🚀🧪

---

Ready to start shipping reliable agents faster? Get a demo and see how LangChain can accelerate your agent engineering lifecycle today. 🎯💡

## Streaming the output from GPT

In [30]:
def stream_generate_brochure(website_name, primary_url):
  llm_stream = llm.chat.completions.create(
    model="gpt-5-nano",
    messages=[
      {"role": "system", "content": brochure_gen_system_prompt},
      {"role": "user", "content": construct_brochure_gen_user_prompt(website_name, primary_url)}
    ],
    stream=True
  )

  response = ""
  display_handle = display(Markdown(""), display_id=True)
  for chunk in llm_stream:
    if chunk.choices[0].finish_reason != None:
      break
    response += chunk.choices[0].delta.content
    display_handle.update(Markdown(response))

- `llm_stream` object is an iterator. For each iteration, it yields a new chunk of data, containing a partial response from the LLM (`chunk.choices[0].delta.content`).
- `response` accumulates the full text over time.
- `display_handle` is a Jupyter display object that updates in place (via `display_id=True`), showing the brochure build up live in the notebook cell.
- Each loop iteration appends the new chunk and refreshes the Markdown display, creating a typewriter-like effect.

In [33]:
stream_generate_brochure("Jonas' Portfolio Page", "https://jonas.io/")

Response:
ChatCompletion(id='chatcmpl-DTjfNgi9uTxKnmEzGn1ItUrMFWyGD', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n  "links": [\n    { "name": "Home - Jonas.io", "url": "https://jonas.io/" },\n    { "name": "Courses", "url": "https://jonas.io/courses" },\n    { "name": "Find Your Course", "url": "https://jonas.io/find-your-course" },\n    { "name": "Resources", "url": "https://jonas.io/resources" },\n    { "name": "AI", "url": "https://jonas.io/ai" },\n    { "name": "AI - Worth Learning", "url": "https://jonas.io/ai#worth-learning" },\n    { "name": "Discord Community", "url": "https://discord.gg/EuDZbyYKhk" },\n    { "name": "Twitter (X) - jonasschmedtman", "url": "https://x.com/jonasschmedtman" },\n    { "name": "GitHub - jonasschmedtmann", "url": "https://github.com/jonasschmedtmann" }\n  ]\n}', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1775979425, model='gpt-

## Jonas Schmedtmann — World-class Web Dev Courses 🚀

Learn to build real products, ship portfolio-worthy apps, and outsmart AI with a mentor trusted by millions.

- 2,000,000+ developers trained worldwide on Udemy 🌍
- 4.7 / 5 average instructor rating from 500k+ reviews ⭐️
- Projects-first, deeply explained, and career-focused 📚

---

### Why learn with Jonas? Why now? 💡

- Clear, detailed explanations with the “why” behind every line of code 🧭
- Polished real-world projects that you can showcase in a portfolio 🧰
- Deep dives into how technologies actually work “under the hood” 🏗️
- Practice through exercises, challenges, and live debugging sessions 🎯
- Teaching that future-proofs your skills in the age of AI 🧠🤖
- Global community and fast, helpful Q&A support 🤝

---

### Core Courses & Library Highlights 📚

All courses are designed, built, and taught by Jonas. They’re available on Udemy and feature hundreds of hours of high-quality content, real-world projects, and lifetime value.

- Build Websites With HTML & CSS
  - 450,000+ students • 38+ hours of video • 2 real-world projects
- The Complete JavaScript Course
  - 1,010,000+ students • 71+ hours • 6 real-world projects
- The Ultimate React Course
  - 150,000+ students • 84+ hours • 10 real-world projects • 275 hours of high-quality content
- The Complete Node.js Bootcamp
  - 170,000+ students • 42+ hours • 2 real-world projects
- Advanced CSS and Sass
  - 220,000+ students • 28+ hours • 3 real-world projects
- Web Dev Crash Course
  - 20,000+ students • 12+ hours • 1 real-world project

- All courses on Udemy with strong student reviews (2,000,000+ total learners on Udemy) 🌟
- Quick-start path options and a dedicated course path finder to help you choose fast 🔎

What you get with every course:
- Risk-free: 30-day money-back guarantee 💸
- Lifetime access and free updates ♾️
- Certificates of completion automatically issued 🎓
- Subtitles in English and 14+ languages 🌍
- Downloadable code, slides, and assets 📦
- 100% supported with fast Q&A responses 🗣️

Access and explore the full library at:
- Home: https://jonas.io/ 🏠
- Courses: https://jonas.io/courses 📘
- Find Your Course: https://jonas.io/find-your-course 🧭

---

### Find Your Course in Minutes — Path Finder 🧭

Generate the perfect course path for your goals in under 3 minutes. Just 4 questions, instant results, no email required, and results stay in your browser. Let’s go! 🚀

- 4-question path builder
- Instant, personalized recommendations
- No email required
- Results stored locally for convenience

---

### AI & Web Development — Stay Ahead 🔮

Jonas shares thoughtful perspectives on AI’s impact on development:
- AI is changing the job of a web developer toward higher-level skills: planning, decision-making, and code-reviewing.
- Learn to collaborate with AI effectively while preserving foundational skills.
- A practical, human-first approach to AI tools and workflows (Cursor, Claude Code, and more) to stay irreplaceable as technology evolves. 🤖➡️🧠

For more, explore:
- AI page: https://jonas.io/ai
- AI: Worth Learning article: https://jonas.io/ai#worth-learning

---

### Community, Support, & Extras 🙌

- Discord community for real-time collaboration and support: https://discord.gg/EuDZbyYKhk 🗨️
- Fast, helpful Q&A support in all courses 🏎️
- Learn from a global network of peers and professionals 🌍

---

### Get Started Today 💥

- Browse the world-class library: https://jonas.io/courses
- Use the Find Your Course path finder: https://jonas.io/find-your-course
- Join the Discord for community support: https://discord.gg/EuDZbyYKhk

Ready to level up your web development skills and build a remarkable portfolio? Let Jonas guide you to become the developer teams want to hire. 🚀🎯

—  
Jonas Schmedtmann © 2015 – 2026 | All rights reserved | Terms & Privacy Policy | AIDiscord community highlights on Resources page